## 🎯 Learning Objectives
* Load and utilize a state-of-the-art pretrained YOLO model for object detection.
* Perform inference on image data using the `ultralytics` library.
* Extract and interpret detection results including bounding boxes, class IDs, and confidence scores.
* Visualize object detection results on an image.


## Exercise: Object Detection with a Pretrained YOLO Model

### Task Description

In this exercise, you will apply your knowledge of advanced computer vision architectures by performing object detection using a pretrained YOLO (You Only Look Once) model. YOLO models are renowned for their speed and accuracy, making them a cornerstone in real-time object detection systems. We will leverage the `ultralytics` library, which provides an easy-to-use interface for various YOLO versions, including the latest YOLOv8.

Your task is to:
1.  **Load a pretrained YOLOv8 model.**
2.  **Perform inference** on a provided sample image.
3.  **Extract the detection results**, including bounding boxes, class labels, and confidence scores.
4.  **Visualize these detections** by drawing bounding boxes and labels on the original image.

### Requirements

*   Use the `ultralytics` library to load and run the YOLO model.
*   The solution should be able to process a given image file.
*   The output should be a visualized image showing detected objects with their bounding boxes and class names.
*   Ensure your code is well-commented and easy to understand.

### Evaluation Criteria

*   **Correctness**: The model should successfully detect common objects in the sample image.
*   **Visualization Quality**: Bounding boxes and labels should be clearly drawn and legible.
*   **Code Clarity**: The code should be clean, efficient, and well-commented.
*   **Adherence to Requirements**: All specified requirements must be met.


In [ ]:
# Install necessary libraries (if not already installed)
# !pip install ultralytics opencv-python matplotlib numpy --quiet

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import os
import requests

print(f"Ultralytics version: {YOLO.__version__}")

# --- Setup: Download a sample image ---
image_url = "https://ultralytics.com/images/bus.jpg"
image_path = "sample_image.jpg"

if not os.path.exists(image_path):
    print(f"Downloading sample image from {image_url}...")
    response = requests.get(image_url, stream=True)
    response.raise_for_status()
    with open(image_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")
else:
    print(f"Sample image '{image_path}' already exists.")

# --- Helper function for displaying images ---
def display_image(image, title="Image", figsize=(10, 10)):
    """Displays an image using matplotlib."""
    plt.figure(figsize=figsize)
    # OpenCV reads images in BGR, Matplotlib expects RGB
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

print("Setup complete. You can now proceed with the exercise.")


### Your Implementation

Now it's your turn! Implement the object detection logic in the code cell below. Follow the steps outlined in the task description:

1.  **Load a pretrained YOLOv8 model.** (Hint: `YOLO('yolov8n.pt')` for the nano version).
2.  **Load the sample image** (`image_path` variable from the setup).
3.  **Perform inference** using the loaded model.
4.  **Process the results**: Iterate through the detections, extract bounding box coordinates, class IDs, and confidence scores.
5.  **Visualize the results**: Draw the bounding boxes and corresponding class labels on a copy of the original image. You can use `cv2.rectangle` and `cv2.putText` for drawing, and the `display_image` helper function to show your final result.


In [ ]:
# --- Solution: Object Detection with Pretrained YOLOv8 ---

# 1. Load a pretrained YOLOv8 model
# We'll use the 'nano' version (yolov8n.pt) for faster inference.
# The model will be downloaded automatically if not present locally.
print("Loading YOLOv8n model...")
model = YOLO('yolov8n.pt')
print("Model loaded successfully.")

# 2. Load the sample image
# cv2.imread loads images in BGR format by default
original_image = cv2.imread(image_path)
if original_image is None:
    raise FileNotFoundError(f"Could not load image from {image_path}. Please check the path.")

# Create a copy of the image to draw detections on
annotated_image = original_image.copy()

# 3. Perform inference on the image
print("Performing inference...")
# The 'results' object contains all detections for the input image(s)
results = model(original_image)
print("Inference complete.")

# 4. Process the results and 5. Visualize the results
# Iterate through each detection result (there might be multiple images in a batch, but here it's just one)
for r in results:
    # 'boxes' contains bounding box coordinates, class IDs, and confidence scores
    # 'names' is a dictionary mapping class IDs to class names
    boxes = r.boxes  # Boxes object for bbox outputs
    names = r.names  # Class names dictionary

    print(f"Detected {len(boxes)} objects.")

    for box in boxes:
        # Get bounding box coordinates
        # xyxy format: [x1, y1, x2, y2]
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Get confidence score
        confidence = box.conf[0].item()

        # Get class ID and name
        class_id = int(box.cls[0].item())
        class_name = names[class_id]

        # Draw bounding box
        # Color for the bounding box (BGR format)
        color = (0, 255, 0) # Green
        thickness = 2
        cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, thickness)

        # Prepare label text
        label = f"{class_name}: {confidence:.2f}"

        # Draw label background (optional, for better readability)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.7
        font_thickness = 2
        text_size = cv2.getTextSize(label, font, font_scale, font_thickness)[0]
        text_x = x1
        text_y = y1 - 10 if y1 - 10 > text_size[1] else y1 + text_size[1] + 10
        
        # Ensure text_y is within image bounds
        if text_y < 0: text_y = 10 + text_size[1]
        if text_y > annotated_image.shape[0]: text_y = annotated_image.shape[0] - 10

        # Draw text background rectangle
        cv2.rectangle(annotated_image, (text_x, text_y - text_size[1] - 5), 
                      (text_x + text_size[0] + 5, text_y + 5), color, -1)

        # Put label text
        cv2.putText(annotated_image, label, (text_x + 5, text_y), 
                    font, font_scale, (0, 0, 0), font_thickness, cv2.LINE_AA)

# Display the annotated image
display_image(annotated_image, title="YOLOv8 Object Detections")

print("Exercise complete. The annotated image has been displayed.")
